# Reviewer 4 (round 2) — fast analyses

Answers two of Reviewer 4's comments with new computations that do not change any existing result:

- **Comment 1** — the *prospective* Experiment B model (14 features, without `NUMBER_OFFERS` and `NUMBER_AWARDS`) evaluated under the temporal split, country-based `GroupKFold` (with fold-specific majority baselines) and the per-country analysis, all in macro-F1.
- **Comment 3** — the if-then rules produced by `convert_to_if_then()` evaluated **as rules**: for each explained instance, the rule's conditions are applied to the held-out fold and we measure *coverage* (share of records that satisfy them) and *precision* (share of those records that receive the same prediction / have the same true label). An optional Anchors (alibi) run is attempted on the same instances as a reference; in our Colab run the alibi installation failed, so it was skipped (see `anchors` in the JSON) and Anchors is not used in the paper.

The LIME setup (fold 10, Random Forest `gini`, explainer and the 50 sampled instances) is identical to `review_1/PROCESO_XAI_figures_EN.ipynb`, so the rules evaluated here are the ones behind the LIME evaluation (Table 23) and the LIME figures of the paper. Results feed Table 20 (Comment 1) and Table 24 (Comment 3).

Expected runtime: about 10–15 minutes without Anchors, 20–30 minutes with it. Results are saved to `reviewer4_fast_results.json`.

In [ ]:
# ── COLAB SETUP ─────────────────────────────────────────────────────────
# Step 1: mount Google Drive
from google.colab import drive
import os
# If already mounted, this line detects it automatically
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

# Step 2: install libraries not included in Colab
!pip install imbalanced-learn lime -q

# Step 3: copy the parquet file from Drive to the local Colab environment
# ─── ADJUST this path to the folder where you uploaded the parquet in your Drive ───
DRIVE_PARQUET = '/content/drive/MyDrive/fci-datos/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
LOCAL_PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
# ────────────────────────────────────────────────────────────────────────────

if not os.path.exists(LOCAL_PARQUET):
    import shutil
    shutil.copy(DRIVE_PARQUET, LOCAL_PARQUET)
    print(f'Parquet copied to {LOCAL_PARQUET}')
else:
    print(f'Parquet already available at {LOCAL_PARQUET}')

print('Setup complete.')

In [ ]:
import pandas as pd, numpy as np, time, json, re, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from imblearn.over_sampling import RandomOverSampler
from lime.lime_tabular import LimeTabularExplainer

df = pd.read_parquet(LOCAL_PARQUET)
for col in df.select_dtypes(include='string').columns:
    df[col] = df[col].astype(object)
for col in ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME', 'YEAR']:
    df[col] = df[col].astype(int)
df['GROUP_CPV'] = df['CPV'].astype(str).str[:2]
df['SME_WIN'] = df['B_CONTRACTOR_SME'].astype(str).str.contains('y', case=False, na=False).astype(int)

cat_cols = ['B_MULTIPLE_CAE', 'B_ON_BEHALF', 'CAE_TYPE', 'MAIN_ACTIVITY',
            'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'GROUP_CPV']
for col in cat_cols:
    df[col] = df[col].astype(str)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(df[cat_cols])
num_cols = ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME']
df_proc = pd.concat([df[num_cols].reset_index(drop=True),
                     pd.DataFrame(ohe_arr, columns=ohe.get_feature_names_out(cat_cols))], axis=1)

uf_A = ['B_MULTIPLE_CAE_n', 'B_ON_BEHALF_n', 'GROUP_CPV_45', 'GROUP_CPV_33', 'GROUP_CPV_15',
        'TYPE_OF_CONTRACT_w', 'ISO_COUNTRY_CODE_si',
        'NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME',
        'CAE_TYPE_3', 'CAE_TYPE_4', 'CAE_TYPE_5', 'ISO_COUNTRY_CODE_lu']
uf_B2 = ['B_MULTIPLE_CAE_n', 'B_ON_BEHALF_n', 'GROUP_CPV_45', 'GROUP_CPV_33', 'GROUP_CPV_15',
         'TYPE_OF_CONTRACT_w', 'MAIN_ACTIVITY_health', 'ISO_COUNTRY_CODE_si',
         'NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS',
         'CAE_TYPE_3', 'MAIN_ACTIVITY_general public\\services',
         'CAE_TYPE_4', 'CAE_TYPE_5', 'ISO_COUNTRY_CODE_lu']
uf_B2_prosp = [f for f in uf_B2 if f not in ('NUMBER_OFFERS', 'NUMBER_AWARDS')]
for f in uf_A + uf_B2:
    assert f in df_proc.columns, f

X_A = df_proc[uf_A].astype(float).to_numpy()
y_A = df['MAIN_ACTIVITY'].astype(str).reset_index(drop=True).to_numpy()
X_B2 = df_proc[uf_B2].astype(float).to_numpy()
X_P = df_proc[uf_B2_prosp].astype(float).to_numpy()
y_B = df['SME_WIN'].reset_index(drop=True).to_numpy()
years = df['YEAR'].reset_index(drop=True).to_numpy()
paises = df['ISO_COUNTRY_CODE'].reset_index(drop=True).to_numpy()
print('X_B2', X_B2.shape, '| X_prospective', X_P.shape)

rf_entropy = RandomForestClassifier(criterion='entropy', max_features='sqrt', bootstrap=True,
                                    max_samples=2/3, n_estimators=100, n_jobs=-1, random_state=42)
ros = RandomOverSampler(random_state=42)
RESULTS = {}

In [ ]:
def fit_predict(model, X_tr, y_tr, X_te):
    m = clone(model)
    Xb, yb = ros.fit_resample(X_tr, y_tr)
    m.fit(Xb, yb)
    return m.predict(X_te), m.predict_proba(X_te)[:, 1]

def scores(y_true, y_pred, y_prob):
    return {'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
            'f1_pos': float(f1_score(y_true, y_pred, zero_division=0)),
            'auc': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else float('nan')}

def majority_f1(y_train, y_test):
    maj = np.bincount(y_train).argmax()
    return float(f1_score(y_test, np.full(len(y_test), maj), average='macro', zero_division=0))

## Comment 1a — Prospective model (14 features), temporal split

In [ ]:
train = years <= 2021
RESULTS['prosp_temporal'] = {}
for label, mask in {'2022': years == 2022, '2023': years == 2023,
                    '2022-2023': np.isin(years, [2022, 2023])}.items():
    p, q = fit_predict(rf_entropy, X_P[train], y_B[train], X_P[mask])
    s = scores(y_B[mask], p, q)
    s['majority_f1_macro'] = majority_f1(y_B[train], y_B[mask])
    RESULTS['prosp_temporal'][label] = s
    print(f'test {label:<10} macro-F1={s["f1_macro"]:.4f}  pos-F1={s["f1_pos"]:.4f}  AUC={s["auc"]:.4f}  majority={s["majority_f1_macro"]:.4f}')

## Comment 1b — Prospective model, country-based GroupKFold (k=5), fold by fold

In [ ]:
rows = []
for i, (tr, te) in enumerate(GroupKFold(n_splits=5).split(X_P, y_B, groups=paises)):
    p, q = fit_predict(rf_entropy, X_P[tr], y_B[tr], X_P[te])
    s = scores(y_B[te], p, q)
    s.update({'fold': i + 1, 'countries': sorted(set(paises[te])), 'n_test': int(len(te)),
              'majority_f1_macro': majority_f1(y_B[tr], y_B[te])})
    rows.append(s)
    print(f'Fold {i+1} {",".join(s["countries"]):<32} macro-F1={s["f1_macro"]:.4f}  AUC={s["auc"]:.4f}  majority={s["majority_f1_macro"]:.4f}')
summ = {k: (float(np.mean([r[k] for r in rows])), float(np.std([r[k] for r in rows])))
        for k in ['f1_macro', 'f1_pos', 'auc', 'majority_f1_macro']}
RESULTS['prosp_country'] = {'per_fold': rows, 'summary': summ}
print('Mean:', {k: f'{v[0]:.4f}±{v[1]:.4f}' for k, v in summ.items()})

## Comment 1c — Prospective model in the seven largest markets (within / pooled incl. / pooled excl.)

In [ ]:
COUNTRIES = ['pl', 'ro', 'fr', 'hu', 'si', 'uk', 'de']
skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_p, oof_q = np.zeros(len(y_B), dtype=int), np.zeros(len(y_B))
for tr, te in skf10.split(X_P, y_B):
    oof_p[te], oof_q[te] = fit_predict(rf_entropy, X_P[tr], y_B[tr], X_P[te])
RESULTS['prosp_pooled_cv'] = scores(y_B, oof_p, oof_q)
print('Pooled prospective OOF:', RESULTS['prosp_pooled_cv'])

rows = []
for c in COUNTRIES:
    m = paises == c
    Xc, yc = X_P[m], y_B[m]
    pa, qa = np.zeros(len(yc), dtype=int), np.zeros(len(yc))
    for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xc, yc):
        pa[te], qa[te] = fit_predict(rf_entropy, Xc[tr], yc[tr], Xc[te])
    pc, qc = fit_predict(rf_entropy, X_P[~m], y_B[~m], Xc)
    r = {'country': c, 'n': int(m.sum()), 'majority_f1_macro': majority_f1(yc, yc),
         'within': scores(yc, pa, qa), 'pooled_included': scores(yc, oof_p[m], oof_q[m]),
         'pooled_excluded': scores(yc, pc, qc)}
    rows.append(r)
    print(f'{c.upper()} AUC within={r["within"]["auc"]:.3f} incl={r["pooled_included"]["auc"]:.3f} excl={r["pooled_excluded"]["auc"]:.3f} | '
          f'F1m within={r["within"]["f1_macro"]:.3f} excl={r["pooled_excluded"]["f1_macro"]:.3f} maj={r["majority_f1_macro"]:.3f}')
RESULTS['prosp_per_country'] = rows

## Comment 3 — Rebuild the exact LIME setup of `review_1/PROCESO_XAI_figures_EN.ipynb` (fold 10, RF `gini`)

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
models, test = {}, {}
for exp_name, X, y in [('A', X_A, y_A), ('B2', X_B2, y_B)]:
    tr_idx, te_idx = list(skf.split(X, y))[9]
    Xb, yb = ros.fit_resample(X[tr_idx], y[tr_idx])
    rf = RandomForestClassifier(n_jobs=-1, max_samples=2/3, random_state=42, n_estimators=100,
                                criterion='gini', max_features='sqrt').fit(Xb, yb)
    models[exp_name] = rf
    test[exp_name] = (X[te_idx], y[te_idx])
X_te_A, y_te_A = test['A']; X_te_B2, y_te_B2 = test['B2']

explainer_A = LimeTabularExplainer(X_te_A, feature_names=uf_A, class_names=list(np.unique(y_te_A)),
                                   mode='classification', random_state=42)
explainer_B2 = LimeTabularExplainer(X_te_B2, feature_names=uf_B2, class_names=['No-SME', 'SME-WIN'],
                                    mode='classification', random_state=42)
np.random.seed(42)
sample_idx = np.random.choice(len(X_te_B2), 50, replace=False)   # same 50 instances as Table 23
print('fold-10 test sizes:', len(X_te_A), len(X_te_B2))

In [ ]:
def parse_condition(cond, feature_names):
    """Turn a LIME/Anchors condition string into (feature_index, lower, upper) with lower < x <= upper."""
    for f in sorted(feature_names, key=len, reverse=True):
        k = cond.find(f)
        if k >= 0:
            break
    else:
        raise ValueError(cond)
    j = feature_names.index(f)
    left, right = cond[:k].strip(), cond[k + len(f):].strip()
    lo, hi = -np.inf, np.inf
    if left:                                   # "a < f <= b"
        lo = float(re.match(r'^(-?[\d.]+)\s*<', left).group(1))
    m = re.match(r'^(<=|>=|<|>|=)\s*(-?[\d.]+)', right)
    if m:
        op, v = m.group(1), float(m.group(2))
        if op in ('<=', '<'):
            hi = v
        elif op in ('>', '>='):
            lo = v
        else:
            lo, hi = v - 1e-9, v
    return j, lo, hi

def rule_mask(conditions, X, feature_names):
    mask = np.ones(len(X), dtype=bool)
    for cond in conditions:
        j, lo, hi = parse_condition(cond, feature_names)
        mask &= (X[:, j] > lo) & (X[:, j] <= hi)
    return mask

def evaluate_rule(conditions, k, X_eval, y_eval, pred_eval, feature_names):
    m = rule_mask(conditions, X_eval, feature_names)
    n = int(m.sum())
    return {'n_conditions': len(conditions), 'coverage': n / len(X_eval),
            'precision_model': float((pred_eval[m] == k).mean()) if n else float('nan'),
            'precision_true': float((y_eval[m] == k).mean()) if n else float('nan'),
            'base_rate_model': float((pred_eval == k).mean())}

### LIME rules as produced by `convert_to_if_then()` (positive-weight conditions, 8 features, as in the paper's LIME figures)

In [ ]:
def lime_rules(exp_name, explainer, model, X_te, y_te, feature_names, idxs, top_labels):
    pred = model.predict(X_te)
    out = []
    for idx in idxs:
        x = X_te[idx]
        k_pred = model.predict(x.reshape(1, -1))[0]
        k_lab = list(model.classes_).index(k_pred)
        e = explainer.explain_instance(x, model.predict_proba, num_features=8, num_samples=1000,
                                       labels=(k_lab,))
        conds = [c for c, w in e.as_list(label=k_lab) if w > 0]      # exactly what convert_to_if_then() verbalizes
        r = evaluate_rule(conds, k_pred, X_te, y_te, pred, feature_names)
        r.update({'instance': int(idx), 'predicted': str(k_pred), 'rule': conds, 'lime_r2': float(e.score)})
        out.append(r)
    return out

t0 = time.time()
rules_B = lime_rules('B2', explainer_B2, models['B2'], X_te_B2, y_te_B2, uf_B2, sample_idx, None)
rules_A = lime_rules('A', explainer_A, models['A'], X_te_A, y_te_A, uf_A, sample_idx[:20], None)
print(f'done in {time.time()-t0:.0f}s')

def summarize_rules(rules, name):
    df_r = pd.DataFrame(rules)
    s = {'n_rules': len(df_r),
         'empty_rules': int((df_r['n_conditions'] == 0).sum()),
         'conditions_mean': float(df_r['n_conditions'].mean()),
         'coverage_mean': float(df_r['coverage'].mean()), 'coverage_median': float(df_r['coverage'].median()),
         'precision_model_mean': float(df_r['precision_model'].mean()),
         'precision_model_median': float(df_r['precision_model'].median()),
         'share_precision_ge_0.95': float((df_r['precision_model'] >= 0.95).mean()),
         'precision_true_mean': float(df_r['precision_true'].mean()),
         'base_rate_model_mean': float(df_r['base_rate_model'].mean())}
    print(f'{name}:', {k: round(v, 4) if isinstance(v, float) else v for k, v in s.items()})
    return s

RESULTS['lime_rules'] = {'B': {'summary': summarize_rules(rules_B, 'LIME rules, Exp B'), 'rules': rules_B},
                         'A': {'summary': summarize_rules(rules_A, 'LIME rules, Exp A'), 'rules': rules_A}}
# The worked example in Section 3.9 (highest-probability SME_WIN=1 instance)
probs = models['B2'].predict_proba(X_te_B2)[:, 1]
idx_pos = int(np.where(y_te_B2 == 1)[0][np.argmax(probs[y_te_B2 == 1])])
ex = lime_rules('B2', explainer_B2, models['B2'], X_te_B2, y_te_B2, uf_B2, [idx_pos], None)[0]
RESULTS['lime_rules']['worked_example'] = ex
print('Worked example rule:', ex['rule'], '| coverage', round(ex['coverage'], 4), '| precision', round(ex['precision_model'], 4))

### Optional: Anchors on the same 50 Experiment B instances (skipped in our run; not used in the paper)

In [ ]:
try:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'alibi'], check=True)
    from alibi.explainers import AnchorTabular
    rf_B2 = models['B2']
    anchor = AnchorTabular(lambda Z: rf_B2.predict(Z), feature_names=uf_B2, seed=42)
    anchor.fit(X_te_B2, disc_perc=(25, 50, 75))
    pred_B = rf_B2.predict(X_te_B2)
    anchors = []
    t0 = time.time()
    for n_done, idx in enumerate(sample_idx, 1):
        x = X_te_B2[idx]
        exp = anchor.explain(x, threshold=0.95)
        k = rf_B2.predict(x.reshape(1, -1))[0]
        r = evaluate_rule(list(exp.anchor), k, X_te_B2, y_te_B2, pred_B, uf_B2)
        r.update({'instance': int(idx), 'rule': list(exp.anchor),
                  'alibi_precision': float(exp.precision), 'alibi_coverage': float(exp.coverage)})
        anchors.append(r)
        if n_done % 10 == 0:
            print(f'  {n_done}/50 ({time.time()-t0:.0f}s)')
    RESULTS['anchors'] = {'B': {'summary': summarize_rules(anchors, 'Anchors, Exp B'), 'rules': anchors}}
except Exception as e:
    print('Anchors skipped:', repr(e))
    RESULTS['anchors'] = {'error': repr(e)}

## Save

In [ ]:
with open('reviewer4_fast_results.json', 'w') as fh:
    json.dump(RESULTS, fh, indent=2, default=float)
print('Saved reviewer4_fast_results.json')
try:
    from google.colab import files
    files.download('reviewer4_fast_results.json')
except Exception as e:
    print('Download it from the Files panel:', e)